# Manual Fuzzy Controller: Velocity-Form Dual Loop (FUZZ-01..06)

Hand-tuned incremental Mamdani fuzzy controller (`src/fuzzy_controller.py`) closing the loop on the Phase-1 plant: T-error+rate drives heater/fan_cooler/vent (vent follows cooling), RH-error+rate drives fogger — the shared `{heater, fan_cooler, fogger, vent}` duty dict fed directly to `step()`. Fixed setpoints 25 C / 70 %RH on the seed-7 weather; the run owns a separate controller state (never shared). Velocity form (output is delta-duty, accumulated with conditional-integration freeze) is the integral action — a positional fuzzy PD droops like P-only control.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.greenhouse_model import step, load_weather_profile, daytime_solar, params, relative_humidity, wsat_g_per_kg
from src.weather_profile_generator import generate_weather_profile
from src.fuzzy_controller import fuzzy_step, FUZZY_MFS, LABELS

T_SP, RH_SP = 25.0, 70.0
print("params:", params)
print("labels:", LABELS)

In [ ]:
# Weather input: regenerate the seed-7 profile only if absent, then load.
import os
CSV = "weather/weather_profile.csv"
if not os.path.exists(CSV):
    generate_weather_profile(CSV, seed=7)

t_min, T_out, w_out = load_weather_profile(CSV)
hours = np.arange(0, 1440) / 60.0          # dt = 60 s -> one step per minute
T_out_hr = np.interp(hours, t_min, T_out)
w_out_hr = np.interp(hours, t_min, w_out)

In [ ]:
# Closed loop with the hand-tuned fuzzy controller over the seed-7 disturbance.
# Separate controller state; setpoints pass as function args.
def run_closed_loop(controller_step):
    T, w = 20.0, 0.65 * wsat_g_per_kg(20.0)
    st = None
    T_hist, RH_hist, U_hist = [], [], []
    for i, h in enumerate(hours):
        rh = relative_humidity(w, T)
        act, st = controller_step(T, rh, T_SP, RH_SP, st)
        ws = (T_out_hr[i], w_out_hr[i], daytime_solar(h, params["A_sol"]))
        T, w, rh = step(60.0, act, ws, T, w)
        T_hist.append(T); RH_hist.append(rh); U_hist.append(act)
    return T_hist, RH_hist, U_hist

T_fz, RH_fz, U_fz = run_closed_loop(fuzzy_step)

iae_T = sum(abs(T - T_SP) for T in T_fz) / 60.0
print("fuzzy: T [%.2f, %.2f]  RH [%.1f, %.1f]  T-IAE %.2f C-h" %
      (min(T_fz), max(T_fz), min(RH_fz), max(RH_fz), iae_T))

**Honest structural note:** RH pins at 100 % overnight under fuzzy control too (probe RH [66.3, 100], matching the on/off and PID precedent). The one-sided fogger authority adds moisture but nothing dries except T-slaved vent exchange — saturation is structural, not a tuning defect. Velocity form kills the temperature droop instead: mean |T-error| over the last 6 h is near zero (probe 0.006 C). The MF-shape panel below is the hand-tuned before-reference for the Phase-4 GA.

In [ ]:
# Trace panels plus hand-tuned MF shapes (GA before-reference) in one figure.
from src.fuzzy_controller import tri
SP_T = np.full_like(hours, T_SP)
SP_RH = np.full_like(hours, RH_SP)
fig = plt.figure(figsize=(14, 14))
gs = fig.add_gridspec(5, 6)

ax = fig.add_subplot(gs[0, :])
ax.plot(hours, T_fz, label="inside T (fuzzy)")
ax.plot(hours, T_out_hr, label="outdoor T", linestyle="--")
ax.plot(hours, SP_T, label="setpoint", linestyle=":")
ax.set_ylabel("Temperature (degC)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = fig.add_subplot(gs[1, :])
ax.plot(hours, RH_fz, color="C2", label="inside RH (fuzzy)")
ax.plot(hours, SP_RH, label="setpoint", linestyle=":")
ax.set_ylabel("RH (%)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = fig.add_subplot(gs[2, :])
for key in ("heater", "fan_cooler", "fogger", "vent"):
    ax.plot(hours, [u[key] for u in U_fz], label=key)
ax.set_xlabel("hour of day"); ax.set_ylabel("duty"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

for col, key in enumerate(("eT", "deT", "du")):
    ax = fig.add_subplot(gs[3, 2 * col:2 * col + 2])
    c, wdt = FUZZY_MFS[key]["centers"], FUZZY_MFS[key]["widths"]
    xs = np.linspace(c[0] - wdt[0], c[-1] + wdt[-1], 400)
    for ci, wi, lab in zip(c, wdt, LABELS):
        ax.plot(xs, [tri(x, ci, wi) for x in xs], label=lab)
    ax.set_title(key); ax.legend(fontsize=7); ax.grid(alpha=0.3)
for col, key in enumerate(("eH", "deH", "dv")):
    ax = fig.add_subplot(gs[4, 2 * col:2 * col + 2])
    c, wdt = FUZZY_MFS[key]["centers"], FUZZY_MFS[key]["widths"]
    xs = np.linspace(c[0] - wdt[0], c[-1] + wdt[-1], 400)
    for ci, wi, lab in zip(c, wdt, LABELS):
        ax.plot(xs, [tri(x, ci, wi) for x in xs], label=lab)
    ax.set_title(key); ax.set_xlabel("universe"); ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle("Manual fuzzy controller - 24h closed-loop (25 C / 70 %RH, seed-7)")
plt.tight_layout()
plt.savefig("notebooks/03_fuzzy.png", dpi=120)
plt.show()